# H03 — Random processes: the noise of a mirror on a spring

> **References**
>
> (Therrien 1992), (Papoulis and Pillai 2002), (Kay 1988), (Harris 1978)

## What you do here

The mirror of H02 hangs from its spring and the ground under it shakes at random. The ground is white noise: every sample is a new number and nothing links one sample to the next. The mirror answers through the transfer function $G(f)$ of H02. Its motion is coloured noise: it remembers its past and it rings near $f_{0}$. The noise of a real detector is made in this way. We describe the motion of the mirror in time with the autocorrelation $r_{xx}[k]$ and in frequency with the power spectral density $S_{xx}(f)$. Then we measure both from the one record we have.

The picture below is the whole workflow. The tasks follow it in order. All the code is given. In every task you change the parameters under `# change these`, run the cells again and answer one question on the physics.

> **The symbols**
>
> - **$m$**, **$k_{\mathrm{s}}$**, **$b$** — the mass of the mirror (kg), the spring constant (N/m) and the damping (kg/s), as in H02; the letter $k$ alone is the lag
> - **$\omega_{0}$**, **$Q$**, **$\tau_{0}$**, **$G(f)$** — the frequency, the quality factor, the ring-down time and the transfer function of the mirror, as in H02
> - **$x_{\mathrm{in}}[n]$**, **$x_{\mathrm{out}}[n]$** — the ground and the mirror, in arbitrary units (a.u.), sampled every $\Delta t$ seconds; $n$ counts the samples
> - **$x^{(i)}[n]$** — the sample $n$ of the record $i = 1, \ldots, R$ of the same process; **$N$** is the number of samples in a record
> - **$r_{xx}[k]$** — the autocorrelation of $x$ at the lag $k$, in steps
> - **$S_{xx}(f)$** — the power spectral density of $x$: one sided, in a.u.$^{2}$ per hertz. It is $2\,\Delta t$ times the two-sided density per cycle per sample of lecture 9. $S_{\mathrm{in}}$ and $S_{\mathrm{out}}$ are the densities of the ground and of the mirror
> - **$H[n]$**, **$D$** — a window and the number of samples in one segment of the record

| Part | The question | Tasks |
|---|---|---|
| A · The process and its memory | What is a random process? How far does its memory reach? | 1 and 2 |
| B · Where the power is | How is the power spread in frequency? How are time and frequency one description? | 3 and 4 |
| C · The estimate from one record | What does a window do? How long must a segment be? | 5 and 6 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.lines import Line2D
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

The cell below draws the workflow. Its functions only draw diagrams: you do not need to read them.

In [ ]:
def draw_block(ax, x, y, text, color, w=2.0, h=1.1, strong=True):
    """Draw one labelled box of a diagram.

    Parameters
    ----------
    ax     : matplotlib axes of the diagram (diagram units)
    x, y   : centre of the box
    text   : text inside the box
    color  : key of the course palette, for example 'BLUE'
    w, h   : width and height of the box
    strong : True for a full box, False for a faded one

    Returns
    -------
    None. The box is drawn on ax.
    """
    # a faded box is a part of the workflow that this task does not use
    alpha = 1.0 if strong else 0.3
    box = FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle='round,pad=0.02,rounding_size=0.12',
                         fc=P[color + '_PALE'], ec=P[color], lw=2, alpha=alpha)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=10.5, color=P['INK'], alpha=alpha)


def draw_chain(labels, colors, arrows):
    """Draw a chain of boxes joined by arrows, from left to right.

    Parameters
    ----------
    labels : list of str, the text in each box
    colors : list of str, the palette key of each box
    arrows : list of str, the text above each arrow (one less than the boxes)

    Returns
    -------
    None. The figure is shown.
    """
    n_box = len(labels)
    fig, ax = plt.subplots(figsize=(3.2 * n_box, 1.9))
    # the boxes, 3.2 units apart, and an arrow with its text in each gap
    for i in range(n_box):
        draw_block(ax, 3.2 * i + 1.0, 0, labels[i], colors[i], w=1.9, h=1.3)
    for i in range(n_box - 1):
        ax.annotate('', xy=(3.2 * (i + 1) + 0.05, 0), xytext=(3.2 * i + 1.95, 0),
                    arrowprops=dict(arrowstyle='-|>', color=P['GREY_DARK'], lw=1.8))
        ax.text(3.2 * i + 2.6, 0.2, arrows[i], ha='center', va='bottom', fontsize=9.5, color=P['GREY_DARK'])
    ax.set(xlim=(-0.2, 3.2 * n_box - 1.0), ylim=(-0.9, 1.0))
    ax.axis('off')
    plt.show()


def draw_mirror(ax, y0):
    """Draw the mirror hanging from its spring, as in H02, with the ground at the top.

    Parameters
    ----------
    ax : matplotlib axes of the diagram (diagram units)
    y0 : height of the mirror in the diagram

    Returns
    -------
    None. The drawing is on ax.
    """
    # the support at the top moves with the ground by x_in; the spring k and the damper b carry the mirror
    ax.add_patch(Rectangle((0.2, y0 + 3.92), 2.0, 0.18, fc=P['GREY_LIGHT'], ec=P['GREY_DARK'], hatch='///'))
    ys = np.linspace(y0 + 3.9, y0 + 1.2, 15)
    xs = 1.5 + 0.28 * np.array([0] + [(-1) ** i for i in range(13)] + [0])
    ax.plot(xs, ys, color=P['BLUE'], lw=2)
    ax.plot([0.7, 0.7], [y0 + 3.9, y0 + 1.2], color=P['AMBER'], lw=6, alpha=0.5)
    ax.add_patch(Rectangle((0.3, y0 + 0.3), 1.7, 0.9, fc=P['BLUE_PALE'], ec=P['BLUE'], lw=2))
    ax.text(1.15, y0 + 0.75, 'mirror, $m$', ha='center', va='center', fontsize=10.5)
    ax.text(1.85, y0 + 2.55, '$k_s$', color=P['BLUE'], fontsize=12, ha='left')
    ax.text(0.5, y0 + 2.55, '$b$', color=P['AMBER'], fontsize=12, ha='right')
    # the motion of the ground and of the mirror, with the kind of noise each one is
    ax.annotate('', xy=(2.7, y0 + 4.4), xytext=(2.7, y0 + 3.5), arrowprops=dict(arrowstyle='<->', color=P['BRICK'], lw=1.8))
    ax.text(2.85, y0 + 4.1, '$x_{in}$\nwhite', color=P['BRICK'], fontsize=11, va='center')
    ax.annotate('', xy=(2.7, y0 + 1.2), xytext=(2.7, y0 + 0.3), arrowprops=dict(arrowstyle='<->', color=P['BRICK'], lw=1.8))
    ax.text(2.85, y0 + 0.75, '$x_{out}$\ncoloured', color=P['BRICK'], fontsize=11, va='center')


def draw_workflow():
    """Draw the workflow of the laboratory: the shaken mirror, its two descriptions and the estimate.

    Returns
    -------
    None. The figure is shown.
    """
    fig, ax = plt.subplots(figsize=(12.5, 7.4))
    # top: the mirror on its spring, and the same system as a block with white noise in and coloured noise out
    draw_mirror(ax, 3.6)
    draw_block(ax, 5.6, 7.0, 'ground\n$x_{in}$, white', 'BRICK', w=2.0, h=1.0)
    draw_block(ax, 8.2, 7.0, 'system\n$G(f)$', 'BLUE', w=2.0, h=1.0)
    draw_block(ax, 10.8, 7.0, 'mirror\n$x_{out}$, coloured', 'BRICK', w=2.2, h=1.0)
    for x_start, x_end in [(6.6, 7.2), (9.2, 9.7)]:
        ax.annotate('', xy=(x_end, 7.0), xytext=(x_start, 7.0), arrowprops=dict(arrowstyle='-|>', color=P['GREY_DARK'], lw=1.8))
    ax.text(8.2, 6.15, r'$S_{out}(f) = |G(f)|^2\,S_{in}(f)$', ha='center', fontsize=11.5, color=P['BRICK'])
    # middle: the one record has two descriptions, joined by the Wiener-Khinchin theorem
    draw_block(ax, 5.6, 4.1, 'In time: $r_{xx}[k]$\nhow the samples are linked', 'SAGE', w=3.2, h=1.1)
    draw_block(ax, 11.0, 4.1, 'In frequency: $S_{xx}(f)$\nwhere the power is', 'BRICK', w=3.2, h=1.1)
    ax.annotate('', xy=(9.35, 4.1), xytext=(7.25, 4.1), arrowprops=dict(arrowstyle='<->', color=P['GREY_DARK'], lw=1.8))
    ax.text(8.3, 4.4, 'Wiener–Khinchin', ha='center', fontsize=9.5, color=P['GREY_DARK'])
    for x_end in (5.6, 11.0):
        ax.annotate('', xy=(x_end, 4.7), xytext=(10.0, 6.45), arrowprops=dict(arrowstyle='-|>', color=P['GREY_DARK'], lw=1.3))
    ax.text(7.3, 5.4, 'one record', ha='center', fontsize=10, color=P['GREY_DARK'])
    # bottom: the estimate from one record: cut, window, square, average
    labels = ['record\n$N$ samples', 'segments\nof $D$ samples', 'window\n$H[n]$', '$|\\mathrm{FFT}|^2$\nof each segment', 'average over\nthe segments']
    for i, label in enumerate(labels):
        draw_block(ax, 1.5 + 2.7 * i, 1.1, label, 'AMBER', w=2.2, h=1.1)
        if i > 0:
            ax.annotate('', xy=(1.5 + 2.7 * i - 1.12, 1.1), xytext=(1.5 + 2.7 * i - 1.58, 1.1),
                        arrowprops=dict(arrowstyle='-|>', color=P['GREY_DARK'], lw=1.8))
    ax.annotate('', xy=(11.5, 3.5), xytext=(12.0, 1.7), arrowprops=dict(arrowstyle='-|>', color=P['AMBER'], lw=1.6, ls='--'))
    ax.text(12.35, 2.6, 'estimates', fontsize=10, color=P['AMBER'], ha='left')
    ax.text(0.3, 2.15, 'What we do with the one record we have', fontsize=10, color=P['GREY_DARK'])
    ax.set(xlim=(0, 13.8), ylim=(0.2, 8.2))
    ax.axis('off')
    plt.show()


draw_workflow()

> **How to use this notebook**
>
> Run the cells in order. Every task has the same parts: what we do, the functions of the task, a cell with the parameters under `# change these`, a box **What to change**, a question and a solution. A function is defined in the task where it is first needed. A later task that uses it says so. The random numbers are seeded: a run gives the numbers of the solution. Change one parameter at a time and run the cell again. Answer the question before you open the solution. Every cell runs in less than 20 seconds.

## Part A · The process and its memory

The ground has no memory. The mirror has. We see first what a random process is and then how far its memory reaches.

### Task 1 · Many mirrors, one mirror

**What we do.** The ground is white Gaussian noise: each sample is a new random number with rms $\sigma_{w}$. We shake the mirror of H02 with it. The product $X_{\mathrm{out}}(f) = G(f)\,X_{\mathrm{in}}(f)$ gives the motion $x_{\mathrm{out}}[n]$ sampled every $\Delta t = 0.02$ s. We do it $R$ times with $R$ different grounds. We get $R$ **realisations** $x^{(i)}[n]$ of one process. At a fixed step $n$ the $R$ values are a random variable. Its mean is the **expectation** $\mathcal{E}\{x[n]\}$. The mean along one record is the **time average**. A detector gives one record, so we need the two averages to agree: the process must be **ergodic**.

The ground is Gaussian and the mirror is linear. A sum of Gaussian numbers is Gaussian: the histogram of the mirror at a fixed $n$ is Gaussian too. The first figure shows one ground record, four realisations of the mirror with the expectation and the histogram across the records at $n_{\text{fixed}}$. The second figure compares the two averages.

> **The symbols**
>
> - **$\sigma_{w}$** — the rms of the white ground in one sample, in a.u.
> - **$R$** — the number of records; **$T$** — the length of one record in seconds, so $N = T/\Delta t$
> - **$\mathcal{E}\{\cdot\}$** — the expectation: the average across the records at a fixed $n$

In [ ]:
# the route of this task: many grounds, the mirror, many records, two averages
draw_chain(['$R$ grounds\n$x_{in}^{(i)}[n]$', 'mirror\n$G(f)$', '$R$ records\n$x_{out}^{(i)}[n]$', 'average across $i$\nor along $n$'],
           ['BRICK', 'BLUE', 'BRICK', 'AMBER'], [r'$\times G(f)$', 'one for each', 'two averages'])

In [ ]:
def oscillator_parameters(m, k_spring, b):
    """Return the natural frequency, the quality factor and the decay time of the mirror (as in H02).

    Parameters
    ----------
    m        : float, mass of the mirror (kg)
    k_spring : float, spring constant (N/m)
    b        : float, damping coefficient (kg/s)

    Returns
    -------
    (f0, q, tau0) : f0 natural frequency (Hz), q quality factor (no unit), tau0 ring-down time (s)
    """
    # natural frequency, quality factor and decay time of the amplitude
    f0 = np.sqrt(k_spring / m) / (2 * np.pi)
    q = np.sqrt(k_spring * m) / b
    tau0 = 2 * m / b
    params = (f0, q, tau0)
    return params


def transfer_function(f, m, k_spring, b):
    """Return the transfer function G = X_out / X_in of the mirror (as in H02).

    Parameters
    ----------
    f        : float or array, frequency (Hz)
    m        : float, mass (kg)
    k_spring : float, spring constant (N/m)
    b        : float, damping coefficient (kg/s)

    Returns
    -------
    G : complex, same shape as f, no unit; |G| is the amplitude gain
    """
    # G = w0^2 / (w0^2 - w^2 + i w w0 / Q), written with k_s = m w0^2 and b = m w0 / Q
    w = 2 * np.pi * f
    G = k_spring / (k_spring - m * w ** 2 + 1j * b * w)
    return G


def shaken_mirror(R, N, dt, m, k_spring, b, sigma_w, seed=2026):
    """Return R records of a white ground and of the mirror that it shakes.

    Uses transfer_function. The mirror is found in frequency, as in task 3 of H02: transform of the
    ground, product with G, transform back. The ringing at the end of a record wraps round to its start:
    for a record much longer than tau0 this is negligible and every sample of the mirror is stationary.

    Parameters
    ----------
    R, N     : number of records and of samples in each (no unit)
    dt       : float, time step (s)
    m, k_spring, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)
    sigma_w  : float, rms of the white ground in one sample (a.u.)
    seed     : integer, seed of the random numbers

    Returns
    -------
    (x_in, x_out) : two arrays of shape (R, N), the ground and the mirror (a.u.); row i is the record i
    """
    # the ground: a new Gaussian number at every step of every record
    rng = np.random.default_rng(seed)
    x_in = sigma_w * rng.normal(size=(R, N))
    # to frequency, times the transfer function, back to time
    f = np.fft.rfftfreq(N, dt)
    X_out = transfer_function(f, m, k_spring, b) * np.fft.rfft(x_in, axis=-1)
    x_out = np.fft.irfft(X_out, N, axis=-1)
    records = (x_in, x_out)
    return records

In [ ]:
# change these: the mirror, the ground and the records
m = 1.0            # mass of the mirror (kg)
k_spring = 39.5    # spring constant (N/m): f0 = sqrt(k_s / m) / 2 pi, 1 Hz here
b = 0.63           # damping coefficient (kg/s): Q = sqrt(k_s m) / b, 10 here
sigma_w = 1.0      # rms of the white ground in one sample (a.u.)
dt = 0.02          # time step (s)
R = 400            # number of records
T = 120.0          # length of one record (s)
n_fixed = 1000     # the step at which we look across the records
n_shown = 4        # records drawn in the first figure

# the records of the ground and of the mirror, and the two averages
f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
N = int(T / dt)
x_in, x_out = shaken_mirror(R, N, dt, m, k_spring, b, sigma_w)
expectation = x_out.mean(axis=0)
time_average = x_out.mean(axis=1)
sigma_out = x_out.std()

# the numbers: the spread of the mirror, the Gaussian check at one step and the scatter of the two averages
across = x_out[:, n_fixed]
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   tau0 = {tau0:.2f} s   {R} records of {N} samples")
print(f"rms of the ground {x_in.std():.3f}   rms of the mirror {sigma_out:.3f}")
print(f"at n = {n_fixed}: fraction of the {R} records within 1 sigma {np.mean(np.abs(across) < sigma_out):.2f}"
      f" and within 2 sigma {np.mean(np.abs(across) < 2 * sigma_out):.3f}   (Gaussian: 0.68 and 0.954)")
print(f"scatter of the expectation over n: {expectation.std():.3f}   rms of the mirror / sqrt(R) = {sigma_out / np.sqrt(R):.3f}")
print(f"scatter of the time averages over the records: {time_average.std():.4f}")

In [ ]:
# first figure: one ground record, a few mirror records with the expectation, the histogram across the records
t = np.arange(N) * dt
shown = t < 30
colours = [P['BLUE'], P['SAGE'], P['AMBER'], P['PLUM']]
fig, axs = plt.subplot_mosaic([['g', 'h'], ['x', 'h']], figsize=(11, 6), gridspec_kw={'width_ratios': [3, 1]})
axs['g'].plot(t[shown], x_in[0][shown], color=P['GREY_MID'], lw=0.7)
axs['g'].set(ylabel='ground $x_{in}$ (a.u.)', xticklabels=[], xlim=(0, 30))
for i in range(n_shown):
    axs['x'].plot(t[shown], x_out[i][shown], color=colours[i % 4], lw=0.9, label=f'record {i + 1}')
axs['x'].plot(t[shown], expectation[shown], color=P['BRICK'], lw=1.8, label='expectation')
axs['x'].axvline(n_fixed * dt, color=P['GREY_MID'], lw=1.0, ls=':')
axs['x'].set(xlabel='time (s)', ylabel='mirror $x_{out}$ (a.u.)', xlim=(0, 30))
axs['x'].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=n_shown + 1, frameon=False, fontsize=9)
v = np.linspace(-4 * sigma_out, 4 * sigma_out, 200)
axs['h'].hist(across, bins=25, density=True, color=P['AMBER_LIGHT'], edgecolor=P['AMBER'], orientation='horizontal')
axs['h'].plot(np.exp(-0.5 * (v / sigma_out) ** 2) / (sigma_out * np.sqrt(2 * np.pi)), v, color=P['BRICK'], lw=1.6)
axs['h'].set(xlabel=f'density at $n = {n_fixed}$', ylabel='$x_{out}$ (a.u.)')
axs['h'].yaxis.set_label_position('right')
axs['h'].yaxis.tick_right()
plt.tight_layout()
plt.show()

# second figure: the values of the two averages on the same axes
fig, ax = plt.subplots(figsize=(9, 3.4))
half = 4 * sigma_out / np.sqrt(R)
bins = np.linspace(-half, half, 41)
ax.hist(time_average, bins=bins, density=True, histtype='step', color=P['BLUE'], lw=1.8, label=f'time averages, {R} values')
ax.hist(expectation, bins=bins, density=True, histtype='step', color=P['BRICK'], lw=1.8, label=f'expectation, {N} values')
ax.axvline(0, color=P['GREY_MID'], lw=1.0, ls=':')
ax.set(xlabel='value of the average (a.u.)', ylabel='density')
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. In the first figure compare the grey record of the ground with the blue records of the mirror. Then read the printed lines.
2. `R`: 100 and 1600. Which curve of the second figure gets narrower?
3. `T`: 30 and 480 s. Which curve gets narrower now?
4. `n_fixed` (try 10 and 5000): does the histogram change?
5. `b` (try 0.21 and 2.1 kg/s): how do the rms of the mirror and the scatter of the expectation change?

**Question.** Which average gets narrower when $R$ grows and which when $T$ grows? When can one record give us the expectation?

<details>
<summary>Solution</summary>

For $R = 400$ records of 120 s the mirror has an rms of $0.794$ a.u. and the ground has $1.000$. The grey ground jumps from one sample to the next. The mirror wanders at about 1 Hz: neighbouring samples are linked. At $n = 1000$ the histogram across the records is Gaussian: $0.70$ of the records are within $1\sigma$ and $0.945$ within $2\sigma$, against $0.68$ and $0.954$. At $n = 10$ and $n = 5000$ the fractions are close to the same: the process does not change with $n$. A different $b$ changes the width of the histogram and not its shape.

The expectation scatters over $n$ by $0.038$. This is the rms of the mirror divided by $\sqrt{R}$: $0.794/\sqrt{400} = 0.040$. With $R = 1600$ it falls to $0.017$ and with $R = 100$ it grows to $0.087$. The time averages scatter by $0.0126$ from one record to the next. A larger $R$ does not move it: $0.0131$ for $R = 1600$. A longer record does: $0.026$ for $T = 30$ s and $0.0065$ for $T = 480$ s, which is a fall as $1/\sqrt{T}$. The expectation does not follow $T$. With $b = 0.21$ and $2.1$ kg/s the rms of the mirror is $1.38$ and $0.434$ and the scatter of the expectation is $0.066$ and $0.021$. The scatter of the time averages is $0.0126$ for both: the time average is dominated by the slowest motions of the ground and there the mirror follows the ground, $G = 1$, whatever $b$ is.

**When one record is enough.** The expectation needs many records and a detector has one. Here the process does not change with $n$ (stationary) and each record runs through all its values (ergodic). The time average of one record then tends to the expectation as $1/\sqrt{T}$. A record is long enough when it is long compared with the memory of the process: here $\tau_{0} = 3.17$ s and the record is 38 times longer. A short record is not enough: at $T = 30$ s the time average scatters by $0.026$.

</details>


### Task 2 · How far the memory reaches

**What we do.** Take one record of the mirror and plot each sample $x[n+k]$ against the sample $x[n]$ that is $k$ steps before. If the cloud is a line, $x[n]$ tells us what comes $k$ steps later: the samples are strongly correlated. If the cloud is round, they are not linked. The **autocorrelation** collects this in one number for every lag:

$$
r_{xx}[k] = \mathcal{E}\{x[n]\,x[n+k]\} .
$$

The mean of the mirror is zero. The process is ergodic, so we estimate $r_{xx}[k]$ as the time average of $x[n]\,x[n+k]$ along one long record. At $k = 0$ it is the variance. The figure draws, above, the scatter for three lags and, below, $r_{xx}[k]/r_{xx}[0]$ for the white ground and for the mirror with three values of $b$. The dashed lines are $\pm\,\mathrm{e}^{-k\Delta t/\tau_{0}}$, the ringing envelope of H02.

> **The symbols**
>
> - **$k$** — the lag, in steps; the time distance is $k\,\Delta t$
> - **$r_{xx}[k]$** — the autocorrelation of one record $x$, in a.u.$^{2}$; $r_{xx}[0]$ is its variance

In [ ]:
# the route of this task: one record, a copy shifted by k steps, the average of the product
draw_chain(['one record\n$x[n]$', 'shifted by $k$\n$x[n+k]$', 'average of\n$x[n]\\,x[n+k]$', '$r_{xx}[k]$\nfor every $k$'],
           ['BRICK', 'BRICK', 'AMBER', 'SAGE'], ['shift', 'multiply', 'repeat'])

In [ ]:
def autocorrelation(x, k_max):
    """Return the autocorrelation r_xx[k], k = 0 ... k_max, of one record: the time average of x[n] x[n+k].

    Parameters
    ----------
    x     : array, one record of a process with zero mean (a.u.)
    k_max : int, the largest lag (steps)

    Returns
    -------
    r : array of k_max + 1 values, the autocorrelation (a.u.^2); r[0] is the variance of the record
    """
    # the sums of x[n] x[n+k] for all the lags at once: transform, square, transform back (the padding avoids wrap-around)
    N = len(x)
    X = np.fft.rfft(x, 2 * N)
    sums = np.fft.irfft(np.abs(X) ** 2)[:k_max + 1]
    # each lag has N - k pairs
    r = sums / (N - np.arange(k_max + 1))
    return r

In [ ]:
# change these: the mirror, with three dampings, and the lags of the scatter plots
m = 1.0                          # mass of the mirror (kg)
k_spring = 39.5                  # spring constant (N/m): f0 = 1 Hz
b_values = (1.26, 0.63, 0.31)    # damping coefficients (kg/s): Q = 5, 10, 20
b_scatter = 0.63                 # damping of the mirror in the scatter plots (kg/s)
lags_s = (0.05, 0.5, 10.0)       # lags of the three scatter plots (s)
sigma_w = 1.0                    # rms of the white ground in one sample (a.u.)
dt = 0.02                        # time step (s)
T = 1000.0                       # length of the record (s)
lag_max_s = 15.0                 # largest lag drawn in the autocorrelation (s)

# one long record of the ground and of the mirror for every damping; the same ground shakes all of them
N = int(T / dt)
k_max = int(lag_max_s / dt)
lag = np.arange(k_max + 1) * dt
records = [shaken_mirror(1, N, dt, m, k_spring, b, sigma_w) for b in b_values]
x_ground = records[0][0][0]
r_ground = autocorrelation(x_ground, k_max)
r_mirror = [autocorrelation(rec[1][0], k_max) for rec in records]
x_scatter = shaken_mirror(1, N, dt, m, k_spring, b_scatter, sigma_w)[1][0]

# the numbers: the variance and the autocorrelation after half a period, which is the first minimum
print(f"ground: r[0] = {r_ground[0]:.3f} (sigma_w^2 = {sigma_w ** 2:.1f})   r[1]/r[0] = {r_ground[1] / r_ground[0]:.4f}")
print("   b (kg/s)      Q   tau0 (s)   r[0] (a.u.^2)   r(0.5 s)/r[0]")
for b, r in zip(b_values, r_mirror):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
    print(f"  {b:8.2f}  {Q:5.1f}  {tau0:9.2f}  {r[0]:14.3f}  {r[int(0.5 / dt)] / r[0]:13.3f}")
print("scatter plots of the mirror with b = %.2f:" % b_scatter)
for lag_s in lags_s:
    k = int(lag_s / dt)
    rho = np.corrcoef(x_scatter[:-k], x_scatter[k:])[0, 1]
    print(f"  lag {lag_s:5.2f} s: correlation {rho:6.3f}   envelope exp(-k dt / tau0) = {np.exp(-lag_s / (2 * m / b_scatter)):.3f}")

In [ ]:
# top: x[n+k] against x[n] for three lags; bottom: the autocorrelation of the ground and of the mirror, normalised to r[0]
colours = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, axs = plt.subplot_mosaic([['s0', 's1', 's2'], ['acf', 'acf', 'acf']], figsize=(11, 8.5),
                              gridspec_kw={'height_ratios': [1, 1.1]})
half = 3.2 * x_scatter.std()
for name, lag_s in zip(('s0', 's1', 's2'), lags_s):
    k = int(lag_s / dt)
    rho = np.corrcoef(x_scatter[:-k], x_scatter[k:])[0, 1]
    axs[name].plot(x_scatter[:2500], x_scatter[k:k + 2500], '.', color=P['BLUE_MID'], ms=3, alpha=0.5)
    axs[name].set(xlim=(-half, half), ylim=(-half, half), aspect='equal', xlabel='$x[n]$ (a.u.)')
    axs[name].set_title(f'lag $k\\,\\Delta t$ = {lag_s} s, correlation {rho:.2f}', fontsize=10, loc='left')
axs['s0'].set_ylabel('$x[n+k]$ (a.u.)')
axs['acf'].plot(lag, r_ground / r_ground[0], color=P['GREY_MID'], lw=2.0, label='ground (white)')
for b, r, c in zip(b_values, r_mirror, colours):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
    axs['acf'].plot(lag, r / r[0], color=c, lw=1.4, label=f'mirror, $Q$ = {Q:.0f}')
    axs['acf'].plot(lag, np.exp(-lag / tau0), color=c, lw=0.9, ls='--')
    axs['acf'].plot(lag, -np.exp(-lag / tau0), color=c, lw=0.9, ls='--')
axs['acf'].set(xlabel='lag $k\\,\\Delta t$ (s)', ylabel=r'$r_{xx}[k]\,/\,r_{xx}[0]$ (no unit)', xlim=(0, lag_max_s))
axs['acf'].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=4, frameon=False)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. Look at the three clouds, then at the autocorrelation of the ground and of the mirror.
2. `lags_s`: try (0.02, 0.25, 3.0) s. At which lag does the cloud turn from a line to a round cloud? Why does the middle one lean the other way?
3. `b_values`: halve each one. How does the memory of each curve change?
4. `k_spring` (try 10 and 160 N/m): how does the rhythm of the autocorrelation change?

**Question.** The ground has no memory and the mirror has. What does the autocorrelation of each one look like? What sets the length of the memory of the mirror?

<details>
<summary>Solution</summary>

The ground has no memory. Its variance is $r_{xx}[0] = 0.997$, which is $\sigma_{w}^{2}$. The next value is $r_{xx}[1]/r_{xx}[0] = 0.0003$. Its autocorrelation is a spike at $k = 0$ (the grey line). The mirror remembers. Its autocorrelation rings at $f_{0} = 1$ Hz and fades inside the envelope $\mathrm{e}^{-k\Delta t/\tau_{0}}$ with $\tau_{0} = 1.59$, $3.17$ and $6.45$ s for $Q = 5$, $10$ and $20$. After half a swing, 0.5 s, it is at its first minimum: $-0.73$, $-0.85$ and $-0.92$. The variance $r_{xx}[0]$ grows with the memory: $0.303$, $0.596$ and $1.198$.

The three clouds: at the lag $0.05$ s the correlation is $0.97$ and the cloud is a line. At $0.5$ s it is $-0.85$ and the line leans the other way: half a swing later the mirror is on the other side. At $10$ s it is $0.04$ and the cloud is round: 10 s is three times $\tau_{0}$ and the envelope is $0.043$. With the lags $(0.02, 0.25, 3.0)$ s the correlations are $0.99$, $0.10$ and $0.37$. After a quarter of a swing the cloud is round although the envelope is still $0.92$. The sample is strongly linked to the earlier one but not along a straight line. A quarter of a swing later the mirror is at the middle of its motion. Its position is not given by the earlier position. The autocorrelation measures only the straight-line link. If $b$ is halved, $Q = 10$, $20$ and $40$ and $\tau_{0} = 3.17$, $6.35$ and $12.9$ s: the memory doubles. With $k_{\mathrm{s}} = 10$ N/m, $f_{0} = 0.50$ Hz and the first minimum moves from 0.5 s to 1 s.

**What the autocorrelation says.** The ground forgets at once: a spike. The mirror remembers for a time $\tau_{0} = 2Q/\omega_{0} = 2m/b$ and rings at $f_{0}$ while it remembers, as the ringing of H02. The memory is set by the damping and the mass. A high $Q$ remembers longer.

</details>


## Part B · Where the power is

The autocorrelation says how the samples are linked in time. The power spectral density says where the power sits in frequency. We measure it by its definition and then we see that it is the same information.

### Task 3 · The power spectral density of the ground and of the mirror

**What we do.** The power spectral density is the expectation of the squared transform of a record divided by its length. The transform of $N$ samples taken every $\Delta t$ gives, for each frequency, the periodogram

$$
\hat{S}(f) = \frac{2\,\Delta t}{N}\,\left|\sum_{n=0}^{N-1} x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n \Delta t}\right|^{2} .
$$

The factor $2\,\Delta t$ puts it in a.u.$^{2}$ per hertz, one sided. We have $R$ records, so we follow the definition: we average $\hat{S}$ over the records. For the white ground this is flat at $S_{\mathrm{in}} = 2\,\sigma_{w}^{2}\,\Delta t$. For the mirror, lecture 9 gives the density that comes out of a linear system:

$$
S_{\mathrm{out}}(f) = |G(f)|^{2}\,S_{\mathrm{in}}(f) .
$$

The figure draws the averages for the ground and for the mirror with three values of $Q$. The dashed lines are the formula.

> **The symbols**
>
> - **$\hat{S}(f)$** — the periodogram: the squared transform of one record, scaled; the hat means that it is estimated from data

In [ ]:
# the route of this task: R records, the squared transform of each, the average
draw_chain(['$R$ records\n$x^{(i)}[n]$', '$\\hat{S}_i(f)$\n$|\\mathrm{FFT}|^2$ each', 'average\nover $i$', '$S_{xx}(f)$\nvs $|G|^2 S_{in}$'],
           ['BRICK', 'AMBER', 'AMBER', 'SAGE'], ['squared FFT', 'mean', 'compare'])

In [ ]:
def periodogram(x, dt):
    """Return the periodogram of a record or of a set of records, one sided, in a.u.^2 per hertz.

    Parameters
    ----------
    x  : array, shape (N,) or (R, N): the records, sampled every dt (a.u.)
    dt : float, time step (s)

    Returns
    -------
    (f, S) : f frequencies from 0 to 1/(2 dt) (Hz); S the squared transform of every record, 2 dt / N |FFT|^2
    """
    # the transform of every record, squared, and scaled to a density per hertz
    N = x.shape[-1]
    f = np.fft.rfftfreq(N, dt)
    S = 2 * dt / N * np.abs(np.fft.rfft(x, axis=-1)) ** 2
    return f, S


def output_psd(f, m, k_spring, b, S_in):
    """Return the density of the mirror from the density of the ground: S_out = |G|^2 S_in.

    Uses transfer_function from task 1.

    Parameters
    ----------
    f        : array, frequency (Hz)
    m, k_spring, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)
    S_in     : float or array, the density of the ground (a.u.^2 per hertz)

    Returns
    -------
    S_out : array like f, the density of the mirror (a.u.^2 per hertz)
    """
    # the power gain is the square of the amplitude gain
    S_out = np.abs(transfer_function(f, m, k_spring, b)) ** 2 * S_in
    return S_out

In [ ]:
# change these: the mirror, with three dampings, and the records
m = 1.0                          # mass of the mirror (kg)
k_spring = 39.5                  # spring constant (N/m): f0 = 1 Hz
b_values = (1.26, 0.63, 0.31)    # damping coefficients (kg/s): Q = 5, 10, 20
sigma_w = 1.0                    # rms of the white ground in one sample (a.u.)
dt = 0.02                        # time step (s)
R = 200                          # number of records
N = 8192                         # samples in one record: 164 s

# the density of the ground, and of the mirror for every damping, as the average of the periodograms over the records
S_in = 2 * sigma_w ** 2 * dt
f = np.fft.rfftfreq(N, dt)
S_ground = None
S_measured, S_formula = [], []
for b in b_values:
    x_in, x_out = shaken_mirror(R, N, dt, m, k_spring, b, sigma_w)
    S_ground = periodogram(x_in, dt)[1].mean(axis=0)
    S_measured.append(periodogram(x_out, dt)[1].mean(axis=0))
    S_formula.append(output_psd(f, m, k_spring, b, S_in))

# the numbers: the level of the ground, the peak of each mirror and the scatter of the average around the formula
print(f"ground: average of the periodograms {S_ground[1:].mean():.4f}   2 sigma_w^2 dt = {S_in:.4f} a.u.^2/Hz")
print("   Q    S at f0 measured   |G|^2 S_in = Q^2 S_in   f0/Q (Hz)")
for b, S_m, S_f in zip(b_values, S_measured, S_formula):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
    i0 = np.argmin(np.abs(f - f0))
    print(f"{Q:5.1f}  {S_m[i0]:16.3f}  {S_f[i0]:20.3f}  {f0 / Q:10.3f}")
band = (f > 0.3) & (f < 10)
print(f"scatter of the measured curve around the formula, 0.3 to 10 Hz: {np.std(S_measured[0][band] / S_formula[0][band]):.3f}   1 / sqrt(R) = {1 / np.sqrt(R):.3f}")

In [ ]:
# the density of the ground and of the mirror for three Q on the same axes; dashed: the formula
colours = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.loglog(f[1:], S_ground[1:], color=P['GREY_MID'], lw=1.6, label='ground (white)')
for b, S_m, S_f, c in zip(b_values, S_measured, S_formula, colours):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
    ax.loglog(f[1:], S_m[1:], color=c, lw=1.2, label=f'mirror, $Q$ = {Q:.0f}')
    ax.loglog(f[1:], S_f[1:], color=P['INK'], lw=0.9, ls='--')
handles = ax.get_legend_handles_labels()[0]
handles += [Line2D([], [], color=P['INK'], lw=0.9, ls='--', label=r'$|G|^2\,S_{in}$')]
ax.legend(handles=handles, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=5, frameon=False, fontsize=9)
ax.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
ax.set(xlabel='frequency (Hz)', ylabel=r'$S_{xx}(f)$ (a.u.$^2$/Hz)', xlim=(0.05, 25), ylim=(1e-8, 40))
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. Read the level of the ground in the printed line. Then compare the three mirrors at $f_{0}$ and far above it.
2. `R`: 20 and 2000. What happens to the scatter of the measured curves around the dashed ones?
3. `sigma_w`: 2. Which curves move and by how much?
4. `b_values`: try (3.15, 0.63, 0.063). How do the height and the width of the peak follow $Q$?
5. `k_spring` (try 10 and 160 N/m): where is the peak?

**Question.** Why is the density of the ground flat and the density of the mirror peaked? What sets the height and the width of the peak?

<details>
<summary>Solution</summary>

The average of the periodograms of the ground is flat at $0.0400$ a.u.$^{2}$/Hz. This is $2\sigma_{w}^{2}\Delta t = 2 \times 1 \times 0.02$. For the mirror with $Q = 5.0$, $10.0$ and $20.3$ the measured density at $f_{0}$ is $1.04$, $4.16$ and $17.2$. The formula $Q^{2}S_{\mathrm{in}}$ gives $0.99$, $3.97$ and $16.4$. The width at half height is $f_{0}/Q = 0.20$, $0.10$ and $0.049$ Hz. Far above $f_{0}$ the three curves fall together as $(f_{0}/f)^{4}S_{\mathrm{in}}$: at 10 Hz the density is $4\times10^{-6}$. The measured curves scatter around the dashed ones by $0.071$, which is $1/\sqrt{R}$. With $R = 20$ it is $0.221$ and with $R = 2000$ it is $0.022$. With $\sigma_{w} = 2$ the ground moves to $0.160$ and the peaks move up by four, $4.16$, $16.6$ and $68.7$: the level scales as $\sigma_{w}^{2}$ and the shape does not change. With $b = (3.15, 0.63, 0.063)$, $Q = 2.0$, $10.0$ and $99.8$: the peaks are $0.167$, $4.16$ and $408$ (formula $0.159$, $3.97$ and $390$) and the widths are $0.50$, $0.10$ and $0.010$ Hz. With $k_{\mathrm{s}} = 10$ and $160$ N/m the peak is at $0.50$ and $2.0$ Hz.

**Why flat and why peaked.** The ground has no memory, so all frequencies carry the same power: its density is flat. The mirror answers each frequency with the gain $|G|$ of H02, so the power is multiplied by $|G|^{2}$. Below $f_{0}$ the gain is 1 and the mirror follows the ground. At $f_{0}$ the gain is $Q$ and the power is multiplied by $Q^{2}$. Above $\sqrt{2}f_{0}$ the gain is below 1 and the mirror is quieter than the ground: the suspension isolates. The density falls to half of its peak where $|G|$ falls by $\sqrt{2}$, which is the half-power width $f_{0}/Q$ of H02. A higher $Q$ gives a taller and narrower peak.

</details>


### Task 4 · Time and frequency are one description

**What we do.** Lecture 9 states the Wiener–Khinchin theorem: the power spectral density is the Fourier transform of the autocorrelation,

$$
S_{xx}(f) = 2\,\Delta t \sum_{k=-\infty}^{\infty} r_{xx}[k]\,\mathrm{e}^{-\mathrm{i} 2\pi f k \Delta t} .
$$

Here we test it on one long record of the mirror and we get $S_{xx}(f)$ by two routes. **Route 1:** the autocorrelation of the record up to a lag $k_{\max}$, then its transform. **Route 2:** we cut the record into $R$ pieces and average their periodograms: the average over records of task 3, with pieces of one record in place of copies. This is allowed because the process is ergodic. The figure draws the two routes and the formula $|G|^{2}S_{\mathrm{in}}$ on the same axes. Lecture 9 also says that the area under the density is the variance. The cell prints the areas.

**Uses** `shaken_mirror` and `oscillator_parameters` from task 1, `autocorrelation` from task 2, `periodogram` and `output_psd` from task 3.

In [ ]:
# the two routes to the same curve, from one long record
draw_chain(['route 1\none long record', 'autocorrelation\n$r_{xx}[k]$', 'transform\nof $r_{xx}[k]$', '$S_{xx}(f)$'],
           ['BRICK', 'SAGE', 'AMBER', 'SAGE'], ['average', 'sum over $k$', 'result'])
draw_chain(['route 2\none long record', '$R$ pieces\n$N$ samples', '$|\\mathrm{FFT}|^2$\nof each piece', '$S_{xx}(f)$\nthe average'],
           ['BRICK', 'AMBER', 'AMBER', 'SAGE'], ['cut', 'square', 'mean'])

In [ ]:
def psd_from_acf(r, N_fft, dt):
    """Return the density as the Fourier transform of the autocorrelation (Wiener-Khinchin).

    Parameters
    ----------
    r     : array, the autocorrelation r[0] ... r[k_max] (a.u.^2)
    N_fft : int, the length of the transform; sets the frequency step 1 / (N_fft dt); must exceed k_max
    dt    : float, time step (s)

    Returns
    -------
    (f, S) : f frequencies (Hz); S the one-sided density 2 dt (r[0] + 2 sum_k r[k] cos(2 pi f k dt)) (a.u.^2 per hertz)
    """
    # the sum over k > 0 of r[k] cos(...) is the real part of the transform of r; the lags -k add the same again
    cos_sum = np.fft.rfft(r, n=N_fft).real
    S = 2 * dt * (2 * cos_sum - r[0])
    f = np.fft.rfftfreq(N_fft, dt)
    return f, S


def piece_average(x, N_piece, dt):
    """Return the density as the average of the periodograms of the pieces of one long record.

    Uses periodogram from task 3.

    Parameters
    ----------
    x       : array, one long record (a.u.)
    N_piece : int, samples in one piece
    dt      : float, time step (s)

    Returns
    -------
    (f, S, R) : f frequencies (Hz); S the average of the periodograms (a.u.^2 per hertz); R the number of pieces
    """
    # cut the record into R pieces, take the periodogram of each and average
    R = len(x) // N_piece
    pieces = x[:R * N_piece].reshape(R, N_piece)
    f, S_each = periodogram(pieces, dt)
    S = S_each.mean(axis=0)
    result = (f, S, R)
    return result

In [ ]:
# change these: the mirror, the record and the two routes
m = 1.0            # mass of the mirror (kg)
k_spring = 39.5    # spring constant (N/m): f0 = 1 Hz
b = 0.63           # damping coefficient (kg/s): Q = 10
sigma_w = 1.0      # rms of the white ground in one sample (a.u.)
dt = 0.02          # time step (s)
N = 2 ** 20        # samples in the long record: 5.8 hours of 20 ms steps
k_max = 1000       # route 1: the largest lag of the autocorrelation (steps): 20 s
N_piece = 8192     # route 2: samples in one piece (164 s)

# one long record of the mirror
f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
x = shaken_mirror(1, N, dt, m, k_spring, b, sigma_w)[1][0]

# route 1: the autocorrelation of the record, then its transform on the frequency grid of the pieces
r = autocorrelation(x, k_max)
f, S_acf = psd_from_acf(r, N_piece, dt)
# route 2: the average of the periodograms of the pieces
f, S_pieces, R = piece_average(x, N_piece, dt)
# the formula
S_formula = output_psd(f, m, k_spring, b, 2 * sigma_w ** 2 * dt)

# the numbers: the peak by the three ways, and the area under each density against the variance
i0 = np.argmin(np.abs(f - f0))
near = slice(i0 - 2, i0 + 3)
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   tau0 = {tau0:.2f} s   lags up to {k_max * dt:.0f} s = {k_max * dt / tau0:.1f} tau0   {R} pieces")
print(f"S near f0 (mean of 5 bins): route 1 {S_acf[near].mean():.2f}   route 2 {S_pieces[near].mean():.2f}   formula {S_formula[near].mean():.2f} a.u.^2/Hz")
i5 = np.argmin(np.abs(f - 5.0))
print(f"S at 5 Hz:                  route 1 {S_acf[i5]:.1e}   route 2 {S_pieces[i5]:.1e}   formula {S_formula[i5]:.1e} a.u.^2/Hz")
print(f"area under S (a.u.^2): route 1 {np.trapezoid(S_acf, f):.4f}   route 2 {np.trapezoid(S_pieces, f):.4f}   formula {np.trapezoid(S_formula, f):.4f}")
print(f"variance of the record {x.var():.4f}   r[0] {r[0]:.4f}")

In [ ]:
# left: the peak on linear axes; right: the whole curve on log axes; the three curves on the same axes
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11.5, 4.6))
for ax in (axa, axb):
    ax.plot(f[1:], S_pieces[1:], color=P['AMBER'], lw=2.4, label='route 2: average of periodograms')
    ax.plot(f[1:], S_acf[1:], color=P['SAGE'], lw=1.2, label='route 1: transform of $r_{xx}[k]$')
    ax.plot(f[1:], S_formula[1:], color=P['INK'], lw=1.0, ls='--', label=r'formula $|G|^2 S_{in}$')
    ax.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
axa.set(xlabel='frequency (Hz)', ylabel=r'$S_{xx}(f)$ (a.u.$^2$/Hz)', xlim=(0, 3))
axb.set_yscale('log')
axb.set(xlabel='frequency (Hz)', ylabel=r'$S_{xx}(f)$ (a.u.$^2$/Hz)', xlim=(0, 8), ylim=(1e-5, 10))
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. Do the three curves lie on one another near the peak? Read the areas and the variance.
2. `k_max`: 100, then 5000. Route 1 sees only lags up to $k_{\max}\Delta t$. When is that too short? When does it only add noise?
3. `N_piece`: 1024 and 32768. Route 2: which gets noisier and which gets smoother?
4. `b` (try 0.21 and 2.1 kg/s): do the areas follow the variance?

**Question.** Why do the two routes give the same curve? What is the area under the density?

<details>
<summary>Solution</summary>

For $f_{0} = 1.000$ Hz and $Q = 10.0$ the two routes lie on the formula. Near $f_{0}$ (the mean of five points) route 1 gives $4.04$, route 2 gives $3.97$ and the formula gives $3.86$ a.u.$^{2}$/Hz. At 5 Hz the formula is $7.0\times10^{-5}$ and route 2 gives $6.9\times10^{-5}$. Route 1 gives $9\times10^{-5}$. Far from the peak the density is tiny. The errors of the autocorrelation at large lags set a noise floor of the order of $10^{-4}$. This is why the green curve scatters in the right plot. The area under route 1 is $0.6402$ and it is $r_{xx}[0]$. The area under route 2 is $0.6402$ and it is the variance of the record. The formula gives $0.627$: the record is one random sample and its variance is 2 % above the variance of the process.

With $k_{\max} = 100$ (2 s, $0.6\,\tau_{0}$) route 1 sees too little of the memory. The peak is smeared: $1.93$ instead of about $4$ and the density at 5 Hz is $1.3\times10^{-2}$. The area is still $0.6402$. With $k_{\max} = 5000$ the peak is $4.03$ but the density at 5 Hz is $-6\times10^{-4}$: the autocorrelation at long lags is estimated from few pairs and adds noise. With $N_{\text{piece}} = 1024$ route 2 has 1024 pieces and a smooth curve but the frequency step is $0.049$ Hz and the peak is covered by two points: $1.92$ at the point nearest $f_{0}$. With $32768$ there are 32 pieces: the step is $0.0015$ Hz and the curve is noisy ($1/\sqrt{32} = 0.18$). For $b = 0.21$ ($Q = 29.9$) the area is $1.942$ and for $b = 2.1$ ($Q = 3.0$) it is $0.189$: the formula gives $1.881$ and $0.188$. The area follows $Q$.

**Why the routes agree.** The Wiener–Khinchin theorem says that the autocorrelation and the density hold the same information: one is the Fourier transform of the other. Route 2 measures the density by its definition and route 1 measures the memory of the record and transforms it. Both can use one long record because the process is ergodic. The memory $\tau_{0}$ in time is the width $f_{0}/Q$ in frequency: the narrow peak and the slow decay of task 2 are one fact. **The area.** Put $k = 0$ in the inverse of the theorem: the area under the density is $r_{xx}[0]$, the variance. The variance of the mirror is the sum of the power in all its frequencies.

</details>


## Part C · The estimate from one record

A detector gives one record. We cut it, window it and average the periodograms of the segments. We see what the window does and how long the segments must be.

### Task 5 · A strong line and a weak line: the window

**What we do.** A real record of a mirror holds lines on top of its noise: the pickup of the mains or a mechanical mode. We add two to the record of the mirror: a strong line at $f_{\mathrm{strong}}$ and a weak line at $f_{\mathrm{weak}}$, a few times $f_{0}$, where the mirror itself is quiet. Cutting $N$ samples out of a record multiplies it by a rectangle. In frequency the rectangle spreads every line over the band in sidelobes that fall slowly. The strong line then lifts the floor and the weak line sits below it. A **window** $H[n]$ that goes smoothly to zero at both ends lowers the sidelobes. We use the **Hann** window and we divide by its mean square $U$ so that the level stays right:

$$
\hat{S}(f) = \frac{2\,\Delta t}{N\,U}\left|\sum_{n=0}^{N-1} H[n]\,x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n \Delta t}\right|^{2},
\qquad
H[n] = \sin^{2}\!\left(\frac{\pi n}{N}\right),
\qquad
U = \frac{1}{N}\sum_{n=0}^{N-1} H^{2}[n] .
$$

For the rectangle $H[n] = 1$. The lines are put half-way between two frequencies of the transform, the worst case for leakage. The figure draws the two estimates on the same axes. On the right we zoom on the strong line.

**Uses** `shaken_mirror`, `oscillator_parameters` and `transfer_function` from task 1, `output_psd` from task 3.

> **The symbols**
>
> - **$H[n]$** — the window: 1 at every sample for the rectangle and $\sin^{2}(\pi n/N)$ for Hann
> - **$U$** — the mean square of the window, which puts back the power the window removes
> - **$f_{\mathrm{strong}}$**, **$f_{\mathrm{weak}}$**, **$A_{\mathrm{strong}}$**, **$A_{\mathrm{weak}}$** — the frequencies of the lines and their amplitudes, in a.u.

In [ ]:
# the route of this task: the record with two lines, the window, the squared transform
draw_chain(['mirror + two lines\n$x[n]$', 'window\n$H[n]\\,x[n]$', '$|\\mathrm{FFT}|^2$', '$\\hat{S}(f)$\nrectangle or Hann'],
           ['BRICK', 'AMBER', 'AMBER', 'SAGE'], ['multiply', 'square', 'scale by $U$'])

In [ ]:
def window_function(name, N):
    """Return the window H[n] of N samples: the rectangle or Hann.

    Parameters
    ----------
    name : 'rect' for 1 at every sample, 'hann' for sin^2(pi n / N)
    N    : int, number of samples

    Returns
    -------
    H : array of N values between 0 and 1
    """
    # the rectangle keeps every sample; Hann goes smoothly to zero at both ends
    n = np.arange(N)
    H = np.ones(N) if name == 'rect' else np.sin(np.pi * n / N) ** 2
    return H


def windowed_periodogram(x, dt, name, pad=1):
    """Return the periodogram of a record or of a set of records, multiplied by a window first.

    Uses window_function.

    Parameters
    ----------
    x    : array, shape (N,) or (K, N): the segments, sampled every dt (a.u.)
    dt   : float, time step (s)
    name : 'rect' or 'hann'
    pad  : int, the sum is evaluated at pad times more frequencies, by adding zeros to the record;
           1 gives the frequencies 1 / (N dt) apart; a larger value adds no information, it draws the same curve finer

    Returns
    -------
    (f, S) : f frequencies from 0 to 1/(2 dt) (Hz); S the one-sided density of every segment (a.u.^2 per hertz)
    """
    # the window, and its mean square, which puts back the power the window removes
    N = x.shape[-1]
    H = window_function(name, N)
    U = np.mean(H ** 2)
    # the transform of the windowed data, squared and scaled to a density per hertz
    f = np.fft.rfftfreq(pad * N, dt)
    S = 2 * dt / (N * U) * np.abs(np.fft.rfft(H * x, n=pad * N, axis=-1)) ** 2
    return f, S


def record_with_lines(x, dt, f_strong, A_strong, f_weak, A_weak):
    """Return the record x with two sinusoidal lines added.

    Each frequency is moved to half-way between two frequencies of the transform of the record,
    the worst case for leakage.

    Parameters
    ----------
    x                : array, the record of the mirror (a.u.)
    dt               : float, time step (s)
    f_strong, f_weak : frequencies of the two lines (Hz)
    A_strong, A_weak : amplitudes of the two lines (a.u.)

    Returns
    -------
    (y, f_strong, f_weak) : the record with the lines and the two frequencies after the move (Hz)
    """
    # the frequency step of the transform and the two frequencies half-way between its points
    N = len(x)
    df = 1 / (N * dt)
    f_strong = (np.round(f_strong / df) + 0.5) * df
    f_weak = (np.round(f_weak / df) + 0.5) * df
    # the two sinusoids on top of the noise of the mirror
    t = np.arange(N) * dt
    y = x + A_strong * np.sin(2 * np.pi * f_strong * t) + A_weak * np.sin(2 * np.pi * f_weak * t)
    result = (y, f_strong, f_weak)
    return result

In [ ]:
# change these: the mirror and the two lines
m = 1.0              # mass of the mirror (kg)
k_spring = 39.5      # spring constant (N/m): f0 = 1 Hz
b = 0.63             # damping coefficient (kg/s): Q = 10
sigma_w = 1.0        # rms of the white ground in one sample (a.u.)
dt = 0.02            # time step (s)
N = 16384            # samples in the record: 328 s
f_strong = 4.5       # frequency of the strong line (Hz)
A_strong = 200.0     # amplitude of the strong line (a.u.)
f_weak = 3.0         # frequency of the weak line (Hz)
A_weak = 0.03        # amplitude of the weak line (a.u.)

# one record of the mirror, then the two lines on top of it
f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
x = shaken_mirror(1, N, dt, m, k_spring, b, sigma_w)[1][0]
y, f_strong, f_weak = record_with_lines(x, dt, f_strong, A_strong, f_weak, A_weak)

# the two estimates, with the rectangle and with Hann
f, S_rect = windowed_periodogram(y, dt, 'rect')
f, S_hann = windowed_periodogram(y, dt, 'hann')

# the same two estimates on a frequency grid 16 times finer: the shape of the strong line
f_fine, S_rect_fine = windowed_periodogram(y, dt, 'rect', pad=16)
f_fine, S_hann_fine = windowed_periodogram(y, dt, 'hann', pad=16)

# the numbers: the same record without the weak line tells us what the line adds at its frequency
y_without = record_with_lines(x, dt, f_strong, A_strong, f_weak, 0.0)[0]
S_mirror = output_psd(f_weak, m, k_spring, b, 2 * sigma_w ** 2 * dt)
S_line = A_weak ** 2 * N * dt / 2
at_line = np.abs(f - f_weak) < 2 / (N * dt)
print(f"mirror alone at {f_weak:.2f} Hz: {S_mirror:.1e} a.u.^2/Hz   the weak line alone would reach about {S_line:.1e}")
print("window       S without the line   S with the line   factor")
for name, S in (('rect', S_rect), ('hann', S_hann)):
    S_without = windowed_periodogram(y_without, dt, name)[1]
    print(f"{name:9s}  {S_without[at_line].max():18.1e}  {S[at_line].max():16.1e}  {S[at_line].max() / S_without[at_line].max():7.1f}")
# the width of the main lobe of the strong line at half height
for name, S in (('rectangle', S_rect_fine), ('Hann', S_hann_fine)):
    lobe = (np.abs(f_fine - f_strong) < 0.05) & (S > S.max() / 2)
    print(f"{name:9s}: main lobe of the strong line {f_fine[lobe].max() - f_fine[lobe].min():.4f} Hz wide at half height"
          f"   (the step of the transform is {1 / (N * dt):.4f} Hz)")

In [ ]:
# left: the two estimates on the same axes; right: the strong line enlarged
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [2.2, 1]})
axa.semilogy(f, S_rect, color=P['GREY_MID'], lw=1.0, label='rectangle')
axa.semilogy(f, S_hann, color=P['BLUE'], lw=1.0, label='Hann')
axb.semilogy(f_fine, S_rect_fine / S_rect_fine.max(), color=P['GREY_MID'], lw=1.4)
axb.semilogy(f_fine, S_hann_fine / S_hann_fine.max(), color=P['BLUE'], lw=1.4)
axa.axvline(f_weak, color=P['BRICK'], lw=0.9, ls=':')
axa.set(xlabel='frequency (Hz)', ylabel=r'$\hat{S}(f)$ (a.u.$^2$/Hz)', xlim=(0, 10), ylim=(1e-7, 1e8))
handles = axa.get_legend_handles_labels()[0]
handles += [Line2D([], [], color=P['BRICK'], lw=0.9, ls=':', label='weak line')]
axa.legend(handles=handles, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False)
axb.set(xlabel='frequency (Hz)', ylabel='$\\hat{S}$ / its largest value', xlim=(f_strong - 0.03, f_strong + 0.03), ylim=(1e-7, 2))
axb.set_title('the strong line, enlarged', fontsize=10, loc='left')
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. Where is the weak line (dotted)? Compare the floor of the grey and of the blue curve around it. Then compare the width of the two peaks on the right.
2. `A_weak`: 0.3 and 0.003. When do both windows see the line and when does only Hann see it?
3. `A_strong`: 20 and 2000. How does the grey floor move? How does the blue one?
4. `f_weak` (try 4.6, 4.55 and 4.52 Hz): the weak line comes close to the strong one. Which window still sees it?

**Question.** Why does the rectangle hide the weak line? What does the Hann window cost?

<details>
<summary>Solution</summary>

With $A_{\mathrm{strong}} = 200$ and $A_{\mathrm{weak}} = 0.03$ the weak line alone would reach $0.15$ a.u.$^{2}$/Hz at 3 Hz. The mirror alone has $6.2\times10^{-4}$ there. With the rectangle the strong line leaks into every frequency: at 3 Hz, 490 steps from the strong line, the floor is $4.0$. The weak line lifts it only to $5.0$, a factor $1.3$: it is hidden. With Hann the floor is $2.6\times10^{-3}$ and the line lifts it to $9.9\times10^{-2}$, a factor $38$: it is the narrow spike in the left plot. Far from the lines Hann follows the mirror down to $10^{-5}$ and the rectangle is still at $0.1$ at 10 Hz.

With $A_{\mathrm{weak}} = 0.3$ the rectangle sees the line (factor $5.0$). Hann sees it with factor $2800$. With $A_{\mathrm{weak}} = 0.003$ the rectangle gives $1.0$ and Hann gives only $2.2$: the line is at the limit. With $A_{\mathrm{strong}} = 20$ the rectangle floor is $0.056$ and the factor is $4.1$: the line is seen. With $A_{\mathrm{strong}} = 2000$ the floor is $380$ and the factor is $1.0$, while Hann gives $37.7$ as before: the Hann floor does not follow the strong line. With $f_{\mathrm{weak}} = 4.6$ Hz, $0.1$ Hz from the strong line, the rectangle gives $0.9$ and Hann $126$. At $4.55$ Hz Hann gives only $3.0$. At $4.52$ Hz, 6 steps away, neither window sees the line: $0.6$ and $0.2$.

**Why the rectangle hides it.** Cutting the record is a multiplication by a rectangle. In frequency the line is spread by the transform of the rectangle. Its first sidelobe is $5\times10^{-2}$ of the peak and the next ones fall slowly (right plot). The strong line is 6700 times larger in amplitude than the weak one: the sidelobes are above it. **The cost of Hann.** The sidelobes are lower: the first one is $7\times10^{-4}$ and they fall fast. But the main lobe is wider. At half height it is $0.0042$ Hz for Hann and $0.0027$ Hz for the rectangle. Two lines closer than a few steps of the transform cannot be separated, with either window. Only a longer record, with a finer step, separates them.

</details>


### Task 6 · One periodogram, Welch and the choice of $D$

**What we do.** In task 3 we averaged over many records. A detector gives one record of $N$ samples. Its periodogram has two errors. The **bias** is the mean of the estimate minus the true $S_{xx}(f)$. The **variance** is the spread of the estimate around its mean. The periodogram of the whole record has a standard deviation as large as its mean. A longer record does not lower it.

**Welch's method** lowers the variance. We cut the record into $K$ segments of $D$ samples, multiply each by the window $H[n]$, square the transform of each and average the $K$ results. The segments overlap by half. The variance falls as $1/K$. But a segment of $D$ samples smooths the spectrum over a frequency step $1/(D\,\Delta t)$: no detail finer than that survives. This is the bias. The peak of the mirror is $f_{0}/Q$ wide. We cut the same long record with three values of $D$. The figure draws the periodogram of the whole record, the three Welch estimates and the formula on the same axes. On the right we zoom on the resonance.

**Uses** `shaken_mirror` and `oscillator_parameters` from task 1, `periodogram` and `output_psd` from task 3, `windowed_periodogram` from task 5.

> **The symbols**
>
> - **$D$** — the number of samples in one segment; **$K$** — the number of segments
> - **$\hat{S}_{W}(f)$** — the Welch estimate: the average of the windowed periodograms of the $K$ segments

In [ ]:
# the route of this task: the record, segments, window, squared transform, average
draw_chain(['record\n$N$ samples', '$K$ segments\nof $D$ samples', 'window $H[n]$\n$|\\mathrm{FFT}|^2$ each', '$\\hat{S}_W(f)$\naverage'],
           ['BRICK', 'AMBER', 'AMBER', 'SAGE'], ['cut', 'window', 'mean'])

In [ ]:
def welch_psd(x, dt, D, name, overlap):
    """Return the Welch estimate of the density: the average of the windowed periodograms of the segments.

    Uses windowed_periodogram from task 5.

    Parameters
    ----------
    x       : array, one record (a.u.)
    dt      : float, time step (s)
    D       : int, samples in one segment
    name    : 'rect' or 'hann', the window of every segment
    overlap : float from 0 to 0.9, the fraction of a segment shared with the next one

    Returns
    -------
    (f, S, K) : f frequencies (Hz), step 1 / (D dt); S the estimate (a.u.^2 per hertz); K the number of segments
    """
    # the segments start every D (1 - overlap) samples
    step = int(D * (1 - overlap))
    starts = np.arange(0, len(x) - D + 1, step)
    segments = np.stack([x[s:s + D] for s in starts])
    # the windowed periodogram of every segment, then the average
    f, S_each = windowed_periodogram(segments, dt, name)
    S = S_each.mean(axis=0)
    result = (f, S, len(starts))
    return result

In [ ]:
# change these: the mirror, the record and the segments
m = 1.0                    # mass of the mirror (kg)
k_spring = 39.5            # spring constant (N/m): f0 = 1 Hz
b = 0.63                   # damping coefficient (kg/s): Q = 10, so the peak is 0.1 Hz wide
sigma_w = 1.0              # rms of the white ground in one sample (a.u.)
dt = 0.02                  # time step (s)
N = 2 ** 17                # samples in the record: 44 minutes
D_values = (64, 512, 4096) # samples in one segment: the frequency step is 1 / (D dt) = 0.78, 0.098, 0.012 Hz
overlap = 0.5              # fraction of a segment shared with the next one

# one long record of the mirror, its periodogram and the three Welch estimates
f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
x = shaken_mirror(1, N, dt, m, k_spring, b, sigma_w)[1][0]
f_all, S_all = windowed_periodogram(x, dt, 'rect')
estimates = [welch_psd(x, dt, D, 'hann', overlap) for D in D_values]
S_in = 2 * sigma_w ** 2 * dt

# the numbers: the scatter in a smooth band, where only the variance matters, and the height at the peak, where the bias shows
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   f0/Q = {f0 / Q:.3f} Hz   N = {N} samples")
smooth = (f_all > 10) & (f_all < 20)
print(f"whole record, one periodogram: S / formula - 1 has std {np.std(S_all[smooth] / output_psd(f_all[smooth], m, k_spring, b, S_in) - 1):.2f}")
print("     D     K   step 1/(D dt) (Hz)   scatter (10 to 20 Hz)   1/sqrt(K)   peak / formula")
for D, (f, S, K) in zip(D_values, estimates):
    truth = output_psd(f, m, k_spring, b, S_in)
    peak = output_psd(f0, m, k_spring, b, S_in)
    i0 = np.argmin(np.abs(f - f0))
    band = (f > 10) & (f < 20)
    scatter = np.std(S[band] / truth[band] - 1)
    print(f"{D:6d}  {K:5d}  {1 / (D * dt):18.3f}  {scatter:20.3f}  {1 / np.sqrt(K):10.3f}  {S[i0 - 1:i0 + 2].max() / peak:13.2f}")

In [ ]:
# left: the periodogram, three Welch estimates and the formula on the same axes; right: the resonance enlarged
colours = [P['BLUE'], P['SAGE'], P['BRICK']]
f_f = np.geomspace(0.02, 25, 3000)
formula = output_psd(f_f, m, k_spring, b, S_in)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.8))
axa.loglog(f_all[1:], S_all[1:], color=P['GREY_LIGHT'], lw=0.8, label='periodogram of the whole record')
near_f0 = np.abs(f_f - f0) < 0.6
for D, (f, S, K), c in zip(D_values, estimates, colours):
    axa.loglog(f[1:], S[1:], color=c, lw=1.5, label=f'Welch, $D$ = {D}')
    axb.plot(f[1:], S[1:], 'o-', color=c, lw=1.5, ms=3.5, label=f'$D$ = {D}')
axa.plot(f_f, formula, color=P['INK'], lw=1.0, ls='--', label=r'$|G|^2 S_{in}$')
axb.plot(f_f[near_f0], formula[near_f0], color=P['INK'], lw=1.0, ls='--', label=r'$|G|^2 S_{in}$')
axa.set(xlabel='frequency (Hz)', ylabel=r'$\hat{S}(f)$ (a.u.$^2$/Hz)', xlim=(0.05, 25), ylim=(1e-8, 40))
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False, fontsize=9)
axb.set(xlabel='frequency (Hz)', ylabel=r'$\hat{S}(f)$ (a.u.$^2$/Hz)', xlim=(f0 - 0.6, f0 + 0.6), ylim=(0, 1.3 * Q ** 2 * S_in))
axb.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=4, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. Read the printed table: the scatter in the smooth band against $1/\sqrt{K}$ and the height of the peak against the formula. On the right plot which curve has the right peak?
2. `D_values`: try (32, 128, 1024). How do the scatter and the peak move?
3. `overlap`: 0 and 0.75. What changes in the scatter?
4. `N`: $2^{19}$. Which of the two errors can a longer record lower?
5. `b` (try 0.126 kg/s, so $Q = 50$): the peak is 0.02 Hz wide. Which $D$ resolves it now?

**Question.** Why does a short segment give a small scatter and a wrong peak while a long segment gives the right peak and a large scatter? How do we choose $D$?

<details>
<summary>Solution</summary>

The record has $N = 131072$ samples. The peak is $f_{0}/Q = 0.100$ Hz wide. The periodogram of the whole record (grey) scatters around the formula by $1.00$: as much as the density itself. A longer record does not lower it. Welch with $D = 64$, $512$ and $4096$ and half overlap has $K = 4095$, $511$ and $63$ segments. In the smooth band, from 10 to 20 Hz, the scatter is $0.016$, $0.049$ and $0.129$ against $1/\sqrt{K} = 0.016$, $0.044$ and $0.126$. The height of the peak, as a fraction of the true height, is $0.12$, $0.62$ and $1.09$. The step $1/(D\Delta t)$ is $0.78$, $0.098$ and $0.012$ Hz. With $D = 64$ the density is smoothed over eight times the width of the peak and the peak is lost. With $D = 512$ the step is the width of the peak and the peak is still $38\,\%$ too low. With $D = 4096$ the step is about a tenth of the width and the peak is right, within the scatter, $13\,\%$.

With no overlap the scatter is $0.025$, $0.061$ and $0.169$, close to $1/\sqrt{K} = 0.022$, $0.062$ and $0.177$. Half overlap lowers the last one to $0.129$ because the Hann window gives little weight to the edges of a segment. Overlap $0.75$ gives $0.123$: almost nothing more, because the segments are no longer independent. With $N = 2^{19}$ the scatter falls to $0.008$, $0.022$ and $0.060$ but the peak stays at $0.12$, $0.62$ and $1.07$ of the true one. A longer record lowers the variance. It does not move the bias at a fixed $D$. With $D = (32, 128, 1024)$ the peak is $0.08$, $0.18$ and $0.79$. With $b = 0.126$ kg/s, $Q = 50$ and the peak is $0.020$ Hz wide. The three values of $D$ give $0.02$, $0.18$ and $0.74$. A longer segment, $D = 16384$ with a step of $0.003$ Hz, gives $1.26$, at the cost of $K = 15$ and a scatter of $0.26$.

**How we choose $D$.** A short segment gives many segments, so a small variance, but it smooths the density over $1/(D\Delta t)$ and loses the peak: a bias. A long segment keeps the peak and gives few segments, so a large variance. For a fixed record we cannot have both small. The narrowest feature we must resolve, here $f_{0}/Q$, sets $D$: the step must be much smaller than the width. Then $K = N/D$ gives the variance. Only a longer record lowers both errors.

</details>


## What to hand in

The answer to the question of each task, in two or three sentences, with the parameters you changed and what you saw:

1. **Task 1.** Which average narrows with $R$ and which with $T$. When one record gives the expectation.
2. **Task 2.** What the autocorrelation of the ground and of the mirror looks like and what sets the memory of the mirror.
3. **Task 3.** Why the density of the ground is flat and the density of the mirror is peaked. What sets the height and the width of the peak.
4. **Task 4.** Why the two routes give the same curve and what the area under the density is.
5. **Task 5.** Why the rectangle hides the weak line and what the Hann window costs.
6. **Task 6.** How the length of a segment trades bias against variance and how we choose it.

> **Where this goes next**
>
> The noise of the mirror is white noise through a transfer function. Its autocorrelation and its density are one description in time and in frequency. One record gives an estimate of the density with a bias and a variance that the segment length trades against each other. The noise of a detector is described and measured in the same way.

## Software

`numpy` throughout; `np.fft.rfft` gives the transforms of the tasks. The random processes follow (Therrien 1992) and (Papoulis and Pillai 2002) and the spectral estimates (Kay 1988). The window is described in (Harris 1978).

**References**

- Therrien (1992). *Discrete Random Signals and Statistical Signal Processing*.
- Papoulis and Pillai (2002). *Probability, Random Variables and Stochastic Processes*.
- Kay (1988). *Spectral Estimation*.
- Harris (1978). *On the use of windows for harmonic analysis with the discrete Fourier transform*.